# WriteWise — Stage 1 Fine-Tuning on CCC, Version 2

Corrected retraining notebook for ML_PIPELINE §§2–5 and PRD §11.
The original run achieved 76.10% test accuracy / 66.33% macro F1; the proposed target
remains >=90% test accuracy. These fixes do not guarantee that target.

**Retrain both phases from ImageNet weights in a fresh Colab T4 GPU runtime.**
Do not resume the original CCC checkpoint: it was trained with black augmentation borders.
Reuse the existing processed CCC data; keep the official test split untouched.

Changes: white augmentation fill (+1), one canonical label mapping, input validation,
frozen BatchNormalization during fine-tuning, and validation-based checkpoint selection
across both phases. The second run achieved 66.23% test accuracy / 57.97% macro F1.
For run three, the only training-policy change is full-dataset shuffling before
normalization, replacing the 2,000-sample shuffle buffer. Architecture, learning rates,
augmentation strength, frozen BatchNormalization, split and class weighting stay fixed.
Section 7.1 evaluates the saved best checkpoint on clean training and validation images
and saves per-class validation reports without loading test data. This experiment tests
whether incomplete shuffling contributed to the generalization gap; a gain is not guaranteed.

Every run writes into a new `checkpoints/stage1_v2/<run-id>/` directory.
Only the convolutional backbone later carries forward into Stage 2; Stage 1 is not
a validated handwriting-quality scorer.

Run Sections 1–7.1 to train and inspect validation results. Set `RUN_FINAL_TEST = True` only
when the configuration is selected, then run the final evaluation cell once.
Do not choose hyperparameters by repeatedly checking test accuracy.


## 1. Setup & Environment


In [ ]:
# Mount Google Drive for data access and persistent checkpoint saving
from google.colab import drive

drive.mount("/content/drive")

import glob
import json
import os
import uuid
import zipfile
from datetime import datetime, timezone
from pathlib import Path

# Configuration — set checkpoint directory in Drive
DRIVE_DIR = "/content/drive/MyDrive/writewise/training"
RUN_ID = (
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:6]
)
CHECKPOINT_DIR = f"{DRIVE_DIR}/checkpoints/stage1_v2/{RUN_ID}"
RUN_FINAL_TEST = (
    False  # Enable only after choosing the configuration on validation data.
)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# High-performance local caching on Colab local SSD (/content/data/processed):
LOCAL_DATA_DIR = "/content/data/processed"

# Look for processed.zip in common locations (Drive or uploaded directly to Colab)
candidate_zip_paths = [
    "/content/processed.zip",
    f"{DRIVE_DIR}/data/processed.zip",
    f"{DRIVE_DIR}/processed.zip",
    "/content/drive/MyDrive/writewise/data/processed.zip",
    "/content/drive/MyDrive/writewise/processed.zip",
    "/content/drive/MyDrive/processed.zip",
]

zip_found = None
for p in candidate_zip_paths:
    if os.path.exists(p):
        zip_found = p
        break


def resolve_processed_dir(candidates):
    """Select a data root by its split folders, not by substrings in its path."""
    for candidate in candidates:
        root = Path(candidate)
        if root.name == "train":
            root = root.parent
        for option in (root, root / "processed"):
            if all((option / split).is_dir() for split in ("train", "val", "test")):
                return str(option)
    return None


DATA_DIR = resolve_processed_dir([LOCAL_DATA_DIR])
if DATA_DIR is None and zip_found:
    print(f"Extracting {zip_found} to {LOCAL_DATA_DIR}...")
    with zipfile.ZipFile(zip_found, "r") as zip_ref:
        zip_ref.extractall(LOCAL_DATA_DIR)
    DATA_DIR = resolve_processed_dir([LOCAL_DATA_DIR])
if DATA_DIR is None:
    DATA_DIR = resolve_processed_dir([f"{DRIVE_DIR}/data/processed"])
if DATA_DIR is None:
    found_dirs = sorted(
        glob.glob("/content/drive/MyDrive/**/processed", recursive=True)
    )
    found_dirs += sorted(glob.glob("/content/drive/MyDrive/**/train", recursive=True))
    DATA_DIR = resolve_processed_dir(found_dirs)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find processed CCC train/val/test folders. Upload processed.zip or set DRIVE_DIR."
    )

print(f"Active DATA_DIR:       {DATA_DIR}")
print(f"Active CHECKPOINT_DIR: {CHECKPOINT_DIR}")

In [ ]:
import numpy as np
import tensorflow as tf

SEED = 42
INPUT_SIZE = 96
tf.keras.utils.set_random_seed(SEED)

print(f"TensorFlow version: {tf.__version__}")
gpus = tf.config.list_physical_devices("GPU")
print(f"GPU available: {len(gpus) > 0}")
if gpus:
    print(f"Using GPU device: {gpus[0].name}")
else:
    print("WARNING: Select a T4 GPU in Runtime -> Change runtime type before training.")

## 2. Load Dataset (Train & Val Splits)


In [ ]:
# CCC class ordering is shared by all splits and matches classes.json.
# Never independently enumerate the classes present in validation or test.
CLASS_NAMES = list("ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz")
CLASS_TO_IDX = {name: index for index, name in enumerate(CLASS_NAMES)}
metadata_path = Path(DATA_DIR) / "classes.json"
if metadata_path.exists():
    class_metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    if (
        class_metadata["classes"] != CLASS_NAMES
        or class_metadata["class_to_idx"] != CLASS_TO_IDX
    ):
        raise ValueError(
            "classes.json does not match the canonical 52-class CCC mapping"
        )


def load_split(split_dir: str) -> tuple[np.ndarray, np.ndarray, list[str]]:
    """Load compact images and encode labels using the same mapping in every split."""
    split_path = Path(split_dir)
    images_file = split_path / "images.npy"
    labels_file = split_path / "labels.npy"
    if images_file.exists() and labels_file.exists():
        images = np.load(images_file, allow_pickle=False)
        raw_labels = np.load(labels_file, allow_pickle=False)
        if raw_labels.ndim != 1:
            raise ValueError("Labels must be a one-dimensional array")
        labels = [str(label) for label in raw_labels]
    else:
        if images_file.exists() or labels_file.exists():
            raise ValueError("Combined split requires both images.npy and labels.npy")
        samples = sorted(split_path.glob("*.npy"))
        labels = [sample.stem.split("_")[0] for sample in samples]
        images = np.array([np.load(sample, allow_pickle=False) for sample in samples])

    if len(images) != len(labels):
        raise ValueError(f"Image/label count mismatch in {split_path}")
    if not labels:
        raise ValueError(f"Empty split: {split_path}")
    if (
        images.dtype != np.uint8
        or images.ndim != 3
        or images.shape[1:] != (INPUT_SIZE, INPUT_SIZE)
    ):
        raise ValueError(
            f"Expected uint8 images with shape (N, 96, 96), got {images.dtype} {images.shape}"
        )
    unknown = sorted(set(labels) - set(CLASS_TO_IDX))
    if unknown:
        raise ValueError(f"Unknown labels in {split_path}: {unknown}")
    encoded = np.array([CLASS_TO_IDX[label] for label in labels], dtype=np.int32)
    return images, encoded, CLASS_NAMES.copy()


print("Loading training data...")
X_train, y_train, class_names = load_split(f"{DATA_DIR}/train")
print("Loading validation data...")
X_val, y_val, val_classes = load_split(f"{DATA_DIR}/val")
NUM_CLASSES = len(class_names)
train_counts = np.bincount(y_train, minlength=NUM_CLASSES)
val_counts = np.bincount(y_val, minlength=NUM_CLASSES)
if np.any(train_counts == 0):
    raise ValueError(
        f"Training has no samples for: {[class_names[i] for i in np.flatnonzero(train_counts == 0)]}"
    )

print(f"Train: {X_train.shape}, {X_train.nbytes / 1e6:.1f} MB; Val: {X_val.shape}")
print("Class | train | val")
for name, train_count, val_count in zip(class_names, train_counts, val_counts):
    print(f"{name:>5} | {train_count:>5} | {val_count:>3}")
missing_val = [class_names[i] for i in np.flatnonzero(val_counts == 0)]
rare_train = [class_names[i] for i in np.flatnonzero(train_counts < 30)]
if missing_val:
    print(f"WARNING: validation cannot measure these classes: {missing_val}")
if rare_train:
    print(f"WARNING: fewer than 30 training samples for: {rare_train}")

run_config = {
    "version": 2,
    "experiment": "run3_full_shuffle",
    "initial_weights": "imagenet",
    "shuffle_buffer_size": len(y_train),
    "reshuffle_each_iteration": True,
    "shuffle_before_normalization": True,
    "run_id": RUN_ID,
    "seed": SEED,
    "tensorflow_version": tf.__version__,
    "data_dir": DATA_DIR,
    "class_names": class_names,
    "class_to_idx": CLASS_TO_IDX,
    "train_counts": train_counts.tolist(),
    "val_counts": val_counts.tolist(),
    "input_size": INPUT_SIZE,
    "batch_size": 32,
    "augmentation_fill": 1.0,
    "batch_normalization_frozen": True,
    "class_weighting": False,
    "phase_a_learning_rate": 1e-3,
    "phase_b_learning_rate": 1e-5,
    "checkpoint_monitor": "val_loss",
}
with open(f"{CHECKPOINT_DIR}/run_config.json", "w", encoding="utf-8") as config_file:
    json.dump(run_config, config_file, indent=2)

## 3. Streaming Preprocessing & Data Augmentation (ML_PIPELINE §4.2)

Keep images as uint8 in RAM. Shuffle the entire training split before converting to
float32, then normalize and augment with tf.data. The raw shuffle payload is about
302 MiB for 34,344 images; normalized RGB images are never fully buffered. Reshuffle
each epoch so changing letter distributions in the source order do not persist.
White pixels (255) normalize to **+1**; black ink (0) normalizes to **-1**.
All geometric transforms use +1 as their fill value. Augmentation is training-only.


In [ ]:
INPUT_SIZE = 96  # ML_PIPELINE §2.3
BATCH_SIZE = 32

# ML_PIPELINE §4.2: rotation capped at ±15°, slight zoom/translate
augmentation_layer = tf.keras.Sequential(
    [
        tf.keras.layers.RandomRotation(
            factor=15 / 360,  # ±15 degrees ceiling
            fill_mode="constant",
            fill_value=1.0,  # white background in [-1, 1] range
        ),
        tf.keras.layers.RandomZoom(
            height_factor=(-0.1, 0.1),
            width_factor=(-0.1, 0.1),
            fill_mode="constant",
            fill_value=1.0,
        ),
        tf.keras.layers.RandomTranslation(
            height_factor=0.05,
            width_factor=0.05,
            fill_mode="constant",
            fill_value=1.0,
        ),
    ]
)


@tf.function
def transform_sample(image, label):
    """Streamed on-the-fly conversion from (96, 96) uint8 to (96, 96, 3) float32 in [-1, 1]."""
    # Grayscale -> 3-channel (MobileNetV2 RGB requirement)
    image = tf.expand_dims(image, -1)
    image = tf.repeat(image, 3, axis=-1)
    # Normalize to [-1, 1]
    image = (tf.cast(image, tf.float32) / 127.5) - 1.0
    # One-hot encode label
    label_onehot = tf.one_hot(label, NUM_CLASSES)
    return image, label_onehot


def build_dataset(images: np.ndarray, labels: np.ndarray, augment: bool = False):
    """Build a memory-efficient tf.data.Dataset using on-the-fly batch transformation."""
    ds = tf.data.Dataset.from_tensor_slices((images, labels))
    if augment:
        # Full shuffle while images are still compact uint8 arrays.
        ds = ds.shuffle(
            buffer_size=len(labels), seed=SEED, reshuffle_each_iteration=True
        )

    # Streaming transform (only processes 32 samples at a time in RAM)
    ds = ds.map(transform_sample, num_parallel_calls=tf.data.AUTOTUNE)

    ds = ds.batch(BATCH_SIZE)
    if augment:
        ds = ds.map(
            lambda x, y: (augmentation_layer(x, training=True), y),
            num_parallel_calls=tf.data.AUTOTUNE,
        )

    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


print("Building streaming datasets...")
train_ds = build_dataset(X_train, y_train, augment=True)
val_ds = build_dataset(X_val, y_val, augment=False)

for batch_x, batch_y in train_ds.take(1):
    print(
        f"  Sample batch shape: {batch_x.shape} (dtype: {batch_x.dtype}), Labels: {batch_y.shape}"
    )
print("Datasets ready with minimal RAM consumption!")

# A white image must remain white, including the generated borders.
white_probe = tf.ones((4, INPUT_SIZE, INPUT_SIZE, 3), dtype=tf.float32)
tf.debugging.assert_near(augmentation_layer(white_probe, training=True), white_probe)

## 4. MobileNetV2 Architecture (ML_PIPELINE §3)


In [ ]:
# ImageNet-pretrained MobileNetV2 base
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(INPUT_SIZE, INPUT_SIZE, 3),
    include_top=False,
    weights="imagenet",
)

# Classification head
x = tf.keras.layers.GlobalAveragePooling2D()(base_model.output)
output = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = tf.keras.Model(inputs=base_model.input, outputs=output)

print(f"Base model layers: {len(base_model.layers)}")
print(f"Total model parameters: {model.count_params():,}")

## 5. Phase A — Head-Only Training (ML_PIPELINE §4.1)

Freeze the base and train the classification head for up to 10 epochs.
Save and restore the best validation-loss checkpoint before Phase B.


In [ ]:
base_model.trainable = False
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
phase_a_path = f"{CHECKPOINT_DIR}/stage1_phase_a_best.keras"
phase_a_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        phase_a_path, monitor="val_loss", save_best_only=True
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True
    ),
    tf.keras.callbacks.CSVLogger(f"{CHECKPOINT_DIR}/phase_a_history.csv"),
    tf.keras.callbacks.TerminateOnNaN(),
]
print("Phase A: head-only training (base frozen)")
history_a = model.fit(
    train_ds, validation_data=val_ds, epochs=10, callbacks=phase_a_callbacks
)
phase_a_best_loss = float(min(history_a.history["val_loss"]))
if not np.isfinite(phase_a_best_loss) or not Path(phase_a_path).exists():
    raise RuntimeError("Phase A did not produce a finite validation-loss checkpoint")
model.load_weights(phase_a_path)

# Seed the final checkpoint with Phase A; Phase B must improve on it to replace it.
checkpoint_path = f"{CHECKPOINT_DIR}/stage1_best.keras"
model.save(checkpoint_path)
print(f"Best Phase A validation loss: {phase_a_best_loss:.4f}")

## 6. Phase B — Partial Unfreeze & Fine-Tuning (ML_PIPELINE §4.1, §4.3)

Unfreeze the top ~30% of layers, except BatchNormalization. Frozen BN layers use
their saved statistics during training and inference, avoiding an abrupt statistics update.
Keep the flat MobileNetV2 → GAP → Dense architecture compatible with Stage 2.
Continue at 1e-5 with early stopping. Replace the final checkpoint only when validation
loss improves on Phase A. See https://keras.io/guides/transfer_learning/.


In [ ]:
base_model.trainable = True
total_layers = len(base_model.layers)
freeze_until = int(total_layers * 0.7)
for index, layer in enumerate(base_model.layers):
    layer.trainable = index >= freeze_until and not isinstance(
        layer, tf.keras.layers.BatchNormalization
    )

trainable_count = sum(layer.trainable for layer in base_model.layers)
print(
    f"Unfroze {trainable_count}/{total_layers} base layers; all BatchNormalization layers frozen"
)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        checkpoint_path,
        monitor="val_loss",
        save_best_only=True,
        initial_value_threshold=phase_a_best_loss,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.CSVLogger(f"{CHECKPOINT_DIR}/phase_b_history.csv"),
    tf.keras.callbacks.TerminateOnNaN(),
]
print("Phase B: fine-tuning with early stopping")
history_b = model.fit(train_ds, validation_data=val_ds, epochs=50, callbacks=callbacks)
phase_b_best_loss = float(min(history_b.history["val_loss"]))
selected_phase = (
    "B"
    if np.isfinite(phase_b_best_loss) and phase_b_best_loss < phase_a_best_loss
    else "A"
)
print(f"Best Phase B validation loss: {phase_b_best_loss:.4f}")
print(f"Selected phase: {selected_phase}; final checkpoint: {checkpoint_path}")

## 7. Training Summary & Curves


In [ ]:
import matplotlib.pyplot as plt

all_acc = history_a.history["accuracy"] + history_b.history["accuracy"]
all_val_acc = history_a.history["val_accuracy"] + history_b.history["val_accuracy"]
all_loss = history_a.history["loss"] + history_b.history["loss"]
all_val_loss = history_a.history["val_loss"] + history_b.history["val_loss"]
phase_a_epochs = len(history_a.history["accuracy"])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
ax1.plot(all_acc, label="Train")
ax1.plot(all_val_acc, label="Validation")
ax1.axvline(
    x=phase_a_epochs - 0.5,
    color="gray",
    linestyle="--",
    alpha=0.6,
    label="Phase A -> B",
)
ax1.set_title("Model Accuracy")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Accuracy")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Loss plot
ax2.plot(all_loss, label="Train")
ax2.plot(all_val_loss, label="Validation")
ax2.axvline(
    x=phase_a_epochs - 0.5,
    color="gray",
    linestyle="--",
    alpha=0.6,
    label="Phase A -> B",
)
ax2.set_title("Model Loss")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
curves_path = f"{CHECKPOINT_DIR}/training_curves.png"
plt.savefig(curves_path, dpi=150)
plt.show()

print("\nFinal training results:")
print(f"  Best validation accuracy: {max(all_val_acc):.4f}")
print(f"  Best validation loss:     {min(all_val_loss):.4f}")
print(f"  Curves saved to:          {curves_path}")

with open(
    f"{CHECKPOINT_DIR}/training_history.json", "w", encoding="utf-8"
) as history_file:
    json.dump(
        {
            "phase_a": history_a.history,
            "phase_b": history_b.history,
            "selected_phase": selected_phase,
        },
        history_file,
        indent=2,
    )

## 7.1. Clean Training & Validation Diagnostics — No Test Data

Reload the saved best checkpoint across both phases. Evaluate both splits in
inference mode without augmentation or shuffling, so the accuracy gap compares
the same weights under the same preprocessing. Epoch training metrics use
augmented inputs and changing weights, and are not directly comparable.

Save classification reports, confusion matrices (CSV/PNG), and
`diagnostics/validation_summary.json`. Macro metrics cover all 52 canonical
classes; classes absent from validation contribute zero and are listed explicitly.
Compare runs on this unchanged validation split before enabling final evaluation.
This cell never loads the official test split.


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)


def evaluate_clean_split(eval_model, images, labels, split_name, output_dir):
    """Evaluate fixed weights with no augmentation or shuffling; keep all 52 labels."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    clean_ds = build_dataset(images, labels, augment=False)
    # Use the training loss path to retain stable cross-entropy from logits.
    evaluation = eval_model.evaluate(clean_ds, verbose=1, return_dict=True)
    predictions = eval_model.predict(clean_ds, verbose=1)
    predicted_labels = np.argmax(predictions, axis=1)
    label_indices = list(range(NUM_CLASSES))
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predicted_labels,
        labels=label_indices,
        average="macro",
        zero_division=0,
    )
    counts = np.bincount(labels, minlength=NUM_CLASSES)
    metrics = {
        "samples": len(labels),
        "loss": float(evaluation["loss"]),
        "accuracy": float(accuracy_score(labels, predicted_labels)),
        "macro_precision": float(precision),
        "macro_recall": float(recall),
        "macro_f1": float(f1),
        "missing_classes": [class_names[i] for i in np.flatnonzero(counts == 0)],
    }
    report = classification_report(
        labels,
        predicted_labels,
        labels=label_indices,
        target_names=class_names,
        zero_division=0,
    )
    (output_dir / f"{split_name}_classification_report.txt").write_text(
        f"Stage 1 — {split_name} (saved checkpoint, no augmentation)\n"
        f"Accuracy: {metrics['accuracy']:.4f} | Macro F1: {f1:.4f}\n\n{report}",
        encoding="utf-8",
    )
    matrix = confusion_matrix(labels, predicted_labels, labels=label_indices)
    np.savetxt(
        output_dir / f"{split_name}_confusion_matrix.csv",
        matrix,
        delimiter=",",
        fmt="%d",
    )
    fig, ax = plt.subplots(figsize=(16, 14))
    heatmap = ax.imshow(matrix, interpolation="nearest", cmap="Blues")
    fig.colorbar(heatmap, ax=ax)
    ax.set(
        title=f"{split_name}: saved checkpoint accuracy {metrics['accuracy']:.2%}",
        xlabel="Predicted label",
        ylabel="True label",
        xticks=label_indices,
        yticks=label_indices,
    )
    ax.set_xticklabels(class_names, rotation=90, fontsize=6)
    ax.set_yticklabels(class_names, fontsize=6)
    fig.tight_layout()
    fig.savefig(output_dir / f"{split_name}_confusion_matrix.png", dpi=150)
    plt.close(fig)
    print(
        f"{split_name}: loss={metrics['loss']:.4f}, "
        f"accuracy={metrics['accuracy']:.2%}, macro F1={f1:.4f}"
    )
    if metrics["missing_classes"]:
        print(
            f"No examples for {metrics['missing_classes']}; macro metrics include "
            "all classes with zero for unsupported classes."
        )
    return metrics


diagnostic_dir = Path(CHECKPOINT_DIR) / "diagnostics"
print("Reloading the selected checkpoint for clean train/validation diagnostics...")
diagnostic_model = tf.keras.models.load_model(checkpoint_path, compile=False)
diagnostic_model.compile(loss="categorical_crossentropy", metrics=["accuracy"])
train_clean_metrics = evaluate_clean_split(
    diagnostic_model, X_train, y_train, "train_clean", diagnostic_dir
)
val_metrics = evaluate_clean_split(
    diagnostic_model, X_val, y_val, "val", diagnostic_dir
)
gap_pp = 100 * (train_clean_metrics["accuracy"] - val_metrics["accuracy"])
validation_summary = {
    "version": 2,
    "experiment": "run3_full_shuffle",
    "run_id": RUN_ID,
    "checkpoint": checkpoint_path,
    "selected_phase": selected_phase,
    "class_names": class_names,
    "macro_average": "all_classes_including_unsupported",
    "train_clean": train_clean_metrics,
    "val": val_metrics,
    "generalization_gap_percentage_points": gap_pp,
}
(diagnostic_dir / "validation_summary.json").write_text(
    json.dumps(validation_summary, indent=2), encoding="utf-8"
)
print(f"Clean train/validation accuracy gap: {gap_pp:.2f} percentage points")
print(f"Diagnostics saved to: {diagnostic_dir}")
del diagnostic_model

## 8. Final Evaluation on Held-Out Test Set (PRD §11 & ML_PIPELINE §5)

After selecting the configuration using validation only, set `RUN_FINAL_TEST = True`
in setup (change the variable in a new cell if training has already finished; do not
rerun setup). Then execute this cell once. It loads the best checkpoint across both phases,
uses the canonical 52-class mapping, and writes reports into this run's evaluation folder.
Test examples are never used for balancing, training, or checkpoint selection.


In [ ]:
if not RUN_FINAL_TEST:
    raise RuntimeError(
        "Final test is disabled. Select the configuration on validation first, then set RUN_FINAL_TEST = True."
    )

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
)

eval_dir = f"{CHECKPOINT_DIR}/evaluation"
os.makedirs(eval_dir, exist_ok=True)

print("Loading best model checkpoint for evaluation...")
eval_model = tf.keras.models.load_model(checkpoint_path)

print("Loading untouched CCC test set...")
X_test, y_test, test_classes = load_split(f"{DATA_DIR}/test")
print(f"  Test samples: {len(X_test)}")
print(f"  Classes:      {len(test_classes)}")
if len(X_test) != 19133:
    raise ValueError(
        f"Expected the official 19,133-character CCC test set, got {len(X_test)}"
    )


# RAM-safe streamed test dataset (no massive float32 arrays in RAM!)
@tf.function
def test_preprocess(image):
    image = tf.expand_dims(image, -1)
    image = tf.repeat(image, 3, axis=-1)
    return (tf.cast(image, tf.float32) / 127.5) - 1.0


test_ds = tf.data.Dataset.from_tensor_slices(X_test)
test_ds = (
    test_ds.map(test_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(64)
    .prefetch(tf.data.AUTOTUNE)
)

print("Running test set inference...")
test_predictions = eval_model.predict(test_ds, verbose=1)
y_pred = np.argmax(test_predictions, axis=1)

# Metrics
acc = float(accuracy_score(y_test, y_pred))
prec = float(precision_score(y_test, y_pred, average="macro", zero_division=0))
rec = float(recall_score(y_test, y_pred, average="macro", zero_division=0))
f1 = float(f1_score(y_test, y_pred, average="macro", zero_division=0))

passed = acc >= 0.90

print()
print("=" * 65)
print("  STAGE 1 FINAL EVALUATION METRICS (CCC TEST SET - 19,133 SAMPLES)")
print("=" * 65)
print(f"  Overall Accuracy:  {acc:.4f} ({acc * 100:.2f}%)")
print(f"  Macro Precision:   {prec:.4f}")
print(f"  Macro Recall:      {rec:.4f}")
print(f"  Macro F1-Score:    {f1:.4f}")
print("  PRD §11 Target:    >= 90.0%")
print(
    f"  Status:            {'✅ PASS (Target Achieved!)' if passed else '❌ BELOW TARGET'}"
)
print("=" * 65)

# Save Classification Report
report = classification_report(
    y_test,
    y_pred,
    labels=list(range(NUM_CLASSES)),
    target_names=test_classes,
    zero_division=0,
)
with open(f"{eval_dir}/classification_report.txt", "w", encoding="utf-8") as f:
    f.write("Stage 1 Evaluation — Classification Report\n")
    f.write(f"Accuracy: {acc:.4f} | F1: {f1:.4f}\n\n")
    f.write(report)

# Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_pred, labels=list(range(NUM_CLASSES)))
np.savetxt(f"{eval_dir}/confusion_matrix.csv", cm, delimiter=",", fmt="%d")

plt.figure(figsize=(16, 14))
plt.imshow(cm, interpolation="nearest", cmap="Blues")
plt.title(f"Stage 1 Confusion Matrix (Test Accuracy: {acc * 100:.1f}%)", fontsize=14)
plt.colorbar()
ticks = list(range(len(test_classes)))
plt.xticks(ticks, test_classes, rotation=90, fontsize=6)
plt.yticks(ticks, test_classes, fontsize=6)
plt.xlabel("Predicted Label", fontsize=12)
plt.ylabel("True Label", fontsize=12)
plt.tight_layout()
plt.savefig(f"{eval_dir}/confusion_matrix.png", dpi=150)
plt.show()

# Save summary JSON
summary = {
    "version": 2,
    "run_id": RUN_ID,
    "selected_phase": selected_phase,
    "class_names": class_names,
    "checkpoint": checkpoint_path,
    "test_samples": len(X_test),
    "accuracy": round(acc, 6),
    "macro_precision": round(prec, 6),
    "macro_recall": round(rec, 6),
    "macro_f1": round(f1, 6),
    "passes_prd_target": passed,
}
with open(f"{eval_dir}/summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(f"\nAll artifacts saved to Drive at: {eval_dir}")